#  Evaluation Dataset

This notebook builds the evaluation dataset.

## 1. CIF Schema

We evaluate a representative subset of the Customer Information Form rather than the full template. The subset is designed to cover the hard extraction cases that matter in real fact-find calls: scalar values, dates, repeated financial tables, nullable fields, ambiguous ownership, family context, free-form objectives, risk preferences, and sparse estate-planning information.

| Section | Fields | Why included |
|---|---|---|
| Personal details | name, DOB, marital / relationship status, mobile phone / email (if mentioned) | scalar fields, dates, nullable PII |
| Household / dependants | partner name and relationship, children / dependants with age, DOB, and dependency status | nested objects, relationships, family context |
| Employment | status, occupation, employer, start date, desired retirement age / date, tax residency (if mentioned) | normalised strings, dates, domain-specific fields |
| Income | owner, source name, amount, frequency, net / gross | repeated rows, owner attribution, numeric normalisation |
| Expenses | category, owner, name, amount, frequency, essential / discretionary flag | repeated rows, categorisation, spending normalisation |
| Pensions / retirement accounts | owner, account type, provider, current value, contribution, account-reference flag | repeated financial products, product classification, privacy-safe reference handling |
| Savings and investments | owner, account type, provider, current value, cash / invested split | product classification, money values, allocation extraction |
| Loans / mortgages | owner, debt type, provider, monthly cost, outstanding balance, interest rate, no-debt flag | debt fields, absence detection, mortgage context |
| Other assets | owner, description, current value, ownership share | non-standard assets, property / business / collectible extraction |
| Objectives | category, description, target amount / income, target date / age, priority, status / uncertainty | free-form goals mapped to structured data |
| Risk profile / preferences | risk score / label, attitude summary, preferred strategy, key concerns | free-form preference extraction and advisory reasoning |
| Estate planning | will status, power of attorney (if mentioned), notes | sparse but important legal planning context |

### Money normalisation rules

Money values use `MoneyAmount` so exact, approximate, and ranged values can be evaluated consistently. For exact values, set `normalized_amount` and leave bounds null. For ranges such as *between 10 and 11 thousand*, set `lower_bound=10000`, `upper_bound=11000`, `normalized_amount=10500`, and `is_approximate=True`. For approximate values such as *about 60k*, set `normalized_amount` to the stated value, `is_approximate=True`, and keep bounds null unless a range is implied. Always preserve the original phrase in `raw_text` when available.


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from __future__ import annotations

import os
import json
from pathlib import Path

from src.eval.models import (
    CIFExtraction,
    ChildDependant,
    ClientProfile,
    EmploymentDetails,
    EmploymentStatus,
    EstatePlanning,
    EvidenceSpan,
    ExpenseItem,
    FieldEvidence,
    Frequency,
    HouseholdDetails,
    IncomeItem,
    LabeledExample,
    LoanMortgageItem,
    MoneyAmount,
    NetGross,
    ObjectiveItem,
    Owner,
    PensionRetirementItem,
    PersonalDetails,
    RiskProfilePreferences,
    SavingsInvestmentItem,
)

from src.eval.audit import (
    audit_ground_truth_dir,
    find_ground_truth_target_mismatches,
    find_inverse_section_mismatches,
    summarise_validations,
)
from src.eval.config import MODEL_CONFIGS
from src.eval.ground_truth import recover_ground_truth_from_raw
from src.eval.pipeline import (
    generate_dataset_async,
    generate_ground_truths_async,
    generate_transcripts_async,
    validate_examples_async,
)
from src.eval.utils import load_transcript_package
from src.eval.scenarios import build_dataset_specs, summarize_dataset_plan
from src.eval.transcript_audit import audit_transcript_file, audit_transcript_format
from src.eval.transcripts import generate_transcript_for_package
from src.eval.utils import (
    load_failed_raw_output,
    load_ground_truth_package,
    load_labeled_example,
    load_transcript_package,
    missing_ground_truth_specs,
    reset_synthetic_output_dirs,
    save_ground_truth_package,
    save_transcript_package,
)

In [ ]:
from openai import AsyncOpenAI

async_client = AsyncOpenAI(api_key=Path("openai_creds.txt").read_text().strip())

### Schema source

The Pydantic models live in [`src/eval/models.py`](src/eval/models.py). Keeping the schema outside the notebook makes it reusable by the extraction pipeline and keeps this notebook focused on labelling and metrics.


In [5]:
# Smoke test

sample = LabeledExample(
    example_id="synthetic_samuel_smith",
    transcript_path="Synthetic Transcript from HT Neviswealth.txt",
    expected=CIFExtraction(
        has_client2=True,
        client1=ClientProfile(
            personal=PersonalDetails(
                first_name="Samuel",
                last_name="Smith",
                date_of_birth="1993-01-08",
                marital_or_relationship_status="cohabiting",
            ),
            employment=EmploymentDetails(
                status=EmploymentStatus.EMPLOYED,
                occupation="Commercial Finance Director",
                employer="Sunrise Assisted Living",
            ),
        ),
        client2=ClientProfile(
            personal=PersonalDetails(first_name="Megan"),
        ),
        household=HouseholdDetails(
            partner_or_spouse_name="Megan",
            partner_or_spouse_is_client2=True,
            partner_or_spouse_relationship="cohabiting partner",
            children_or_dependants=[
                ChildDependant(name="Maylee", relationship="child", age=3, dependency_status="dependent child in childcare"),
                ChildDependant(name="Lylee", relationship="child", age=1.5, dependency_status="dependent child in childcare"),
            ],
        ),
        incomes=[
            IncomeItem(owner=Owner.CLIENT1, source_name="current salary and car allowance", amount=MoneyAmount(normalized_amount=165000, is_approximate=True, raw_text="$165 000"), frequency=Frequency.ANNUAL, net_or_gross=NetGross.GROSS),
            IncomeItem(owner=Owner.CLIENT1, source_name="bonus", amount=MoneyAmount(normalized_amount=39000, is_approximate=False, raw_text="$39,000"), frequency=Frequency.ANNUAL, net_or_gross=NetGross.GROSS),
        ],
        expenses=[
            ExpenseItem(category="childcare", owner=Owner.HOUSEHOLD, name="childcare", amount=None, frequency=None, is_essential=True),
        ],
        pensions_and_retirement_accounts=[
            PensionRetirementItem(owner=Owner.CLIENT1, provider="Vanguard", account_type="Solo 401(k)", current_value=MoneyAmount(normalized_amount=61000, is_approximate=False, raw_text="$61,000")),
            PensionRetirementItem(owner=Owner.CLIENT1, provider="BrightPath", account_type="Workplace 401(k)", current_value=MoneyAmount(normalized_amount=500, is_approximate=True, raw_text="$500"), contribution="3%"),
        ],
        savings_and_investments=[
            SavingsInvestmentItem(owner=Owner.HOUSEHOLD, account_type="high-yield savings", current_value=MoneyAmount(normalized_amount=130000, is_approximate=True, raw_text="$130,000"), cash_value=MoneyAmount(normalized_amount=130000, is_approximate=True, raw_text="$130,000")),
            SavingsInvestmentItem(owner=Owner.HOUSEHOLD, account_type="investment fund", current_value=MoneyAmount(normalized_amount=26000, is_approximate=False, raw_text="$26,000"), invested_value=MoneyAmount(normalized_amount=26000, is_approximate=False, raw_text="$26,000")),
        ],
        loans_and_mortgages=[
            LoanMortgageItem(owner=Owner.HOUSEHOLD, debt_type="mortgage", no_debt=False),
        ],
        objectives=[
            ObjectiveItem(category="tax efficiency", description="Keep taxable income below the childcare benefit threshold by increasing retirement contributions.", target_amount_or_income=MoneyAmount(normalized_amount=150000, is_approximate=False, raw_text="$150,000"), priority="high"),
            ObjectiveItem(category="investment growth", description="Start investing part of excess savings while keeping sufficient cash reserves.", priority="medium"),
        ],
        risk_profile_and_preferences=RiskProfilePreferences(
            risk_score_or_label="medium degree of risk",
            attitude_summary="Prepared to take medium risk for longer-term investment performance.",
            key_concerns=["market volatility", "childcare benefit threshold"],
        ),
        estate_planning=EstatePlanning(will_status="not mentioned"),
    ),
    evidence=[
        FieldEvidence(
            field_path="client1.personal.date_of_birth",
            evidence=[EvidenceSpan(quote="January 8th, 1993", speaker="CLIENT", timestamp="0:12:25")],
            confidence=1.0,
        )
    ],
)

print("Schema valid")
print(f"Top-level CIF fields: {list(CIFExtraction.model_fields.keys())}")
print(json.dumps(sample.model_dump(mode="json", exclude_none=True), indent=2))


Schema valid
Top-level CIF fields: ['has_client2', 'client1', 'client2', 'household', 'incomes', 'expenses', 'pensions_and_retirement_accounts', 'savings_and_investments', 'loans_and_mortgages', 'other_assets', 'objectives', 'risk_profile_and_preferences', 'estate_planning']
{
  "example_id": "synthetic_samuel_smith",
  "transcript_path": "Synthetic Transcript from HT Neviswealth.txt",
  "expected": {
    "has_client2": true,
    "client1": {
      "personal": {
        "first_name": "Samuel",
        "last_name": "Smith",
        "date_of_birth": "1993-01-08",
        "marital_or_relationship_status": "cohabiting"
      },
      "employment": {
        "status": "employed",
        "occupation": "Commercial Finance Director",
        "employer": "Sunrise Assisted Living"
      }
    },
    "client2": {
      "personal": {
        "first_name": "Megan"
      },
      "employment": {}
    },
    "household": {
      "partner_or_spouse_name": "Megan",
      "partner_or_spouse_is_client2"

## 2. Synthetic Dataset Design

The quantitative evaluation dataset contains 68 synthetic fact-find transcripts with structured ground truth labels. The two provided transcripts are kept as qualitative holdout examples and are not used to tune prompts or metrics.


The dataset-generation utilities live in `src/eval/config.py`, `scenarios.py`, `ground_truth.py`, `transcripts.py`, `evaluation.py`, `pipeline.py`, and `utils.py`. This keeps the notebook focused on configuration review, execution, and analysis.


In [ ]:
specs = build_dataset_specs(seed=42)
summary = summarize_dataset_plan(specs)
summary


{'total': 80,
 'by_difficulty': {'easy': 20, 'medium': 30, 'hard': 30},
 'by_archetype': {'retired_widowed_client': 10,
  'dual_income_mortgage_household': 10,
  'high_debt_low_savings': 10,
  'inheritance_windfall': 14,
  'pre_retirement_couple': 13,
  'self_employed_single': 5,
  'messy_corrections_privacy': 10,
  'young_high_earner_family': 8},
 'by_tag': {'advisor_noise': 36,
  'client2_present': 46,
  'correction': 40,
  'date_normalization': 12,
  'estate_planning': 37,
  'joint_assets': 38,
  'missing_fields': 34,
  'multiple_products': 25,
  'negation': 39,
  'numeric_approximate': 43,
  'numeric_exact': 41,
  'numeric_range': 32,
  'numeric_shorthand': 32,
  'objectives_free_form': 44,
  'owner_attribution': 44,
  'privacy_reference': 20,
  'risk_preferences': 36},
 'by_section': {'employment': 70,
  'estate_planning': 26,
  'expenses': 74,
  'household_dependants': 65,
  'income': 80,
  'loans_mortgages': 58,
  'objectives': 80,
  'other_assets': 21,
  'pensions_retirement': 

In [ ]:
MODEL_CONFIGS

{'ground_truth': {'model': 'gpt-5.4-mini',
  'temperature': 0.1,
  'reasoning': {'effort': 'none'},
  'timeout_seconds': 240.0},
 'transcript': {'model': 'gpt-5.4-mini',
  'temperature': 0.7,
  'reasoning': {'effort': 'none'},
  'timeout_seconds': 600.0},
 'validation': {'model': 'gpt-5.4',
  'temperature': None,
  'reasoning': {'effort': 'high'},
  'timeout_seconds': 600.0}}

In [ ]:
reset_synthetic_output_dirs()
specs = build_dataset_specs(seed=42)

In [12]:
ground_truth_packages, gt_failures = await generate_ground_truths_async(
    async_client,
    specs,
    max_concurrency=4,
    save=True,
    show_progress=True,
)

gt_failures, ground_truth_packages

ground_truth:   0%|          | 0/80 [00:00<?, ?example/s]

([],
 [{'example_id': 'easy_001_retired_widowed_client',
   'difficulty': 'easy',
   'archetype_id': 'retired_widowed_client',
   'challenge_tags': ['negation', 'numeric_approximate', 'numeric_exact'],
   'section_targets': ['personal_details',
    'household_dependants',
    'income',
    'expenses',
    'pensions_retirement',
    'savings_investments',
    'loans_mortgages',
    'objectives',
    'risk_profile_preferences'],
   'age_band': 'retired_1948_1962',
   'include_mobile_phone': True,
   'include_email': False,
   'expected': {'has_client2': False,
    'client1': {'personal': {'first_name': 'Leila',
      'last_name': 'Delgado',
      'date_of_birth': '1956-04-18',
      'marital_or_relationship_status': 'widowed',
      'mobile_phone': '+1-415-555-0187',
      'email': None},
     'employment': {'status': 'retired',
      'occupation': 'retired teacher',
      'employer': None,
      'start_date': None,
      'desired_retirement_age': None,
      'desired_retirement_date': N

In [ ]:
audit_ground_truth_dir()

{'total_files': 80,
 'mobile_phone_fields': 32,
 'email_fields': 26,
 'has_client2': 63,
 'client1_dob_missing': 0,
 'dob_year_min': 1955,
 'dob_year_max': 2001,
 'dob_year_buckets': {1950: 12,
  1960: 30,
  1970: 15,
  1980: 42,
  1990: 34,
  2000: 7},
 'dob_after_2000': 7,
 'dob_before_1960': 12}

In [ ]:
ground_truth_target_mismatches = find_ground_truth_target_mismatches()
len(ground_truth_target_mismatches), ground_truth_target_mismatches[:10]


(0, [])

In [23]:
from src.eval.config import GROUND_TRUTH_DIR

target_mismatches = find_ground_truth_target_mismatches()
bad_example_ids = {row["example_id"] for row in target_mismatches}

bad_specs = [spec for spec in specs if spec.example_id in bad_example_ids]

len(bad_specs), sorted(bad_example_ids)


(0, [])

In [ ]:
for example_id in bad_example_ids:
    path = GROUND_TRUTH_DIR / f"{example_id}.json"
    if path.exists():
        path.unlink()

regenerated_gt_packages, regenerated_gt_failures = await generate_ground_truths_async(
    async_client,
    bad_specs,
    max_concurrency=2,
    save=True,
    show_progress=True,
)

len(regenerated_gt_packages), regenerated_gt_failures


ground_truth:   0%|          | 0/7 [00:00<?, ?example/s]

(7, [])

In [25]:
target_mismatches_after = find_ground_truth_target_mismatches()
len(target_mismatches_after), target_mismatches_after[:10]

(0, [])

### Regenerate ground truths with objective-section conflicts
Find files where an objective references `estate_planning` (or another section)
but that section is null/empty in the label. The transcript generator will
discuss those topics anyway (to support the objective), producing extractable
facts that the null ground truth cannot validate. Regenerate with the fixed
prompt that enforces section-objective consistency.

In [49]:
from src.eval.audit import find_objective_section_conflicts

obj_conflicts = find_objective_section_conflicts()
print(f"{len(obj_conflicts)} ground truths with objective-section conflicts")
obj_conflicts

17 ground truths with objective-section conflicts


[{'example_id': 'easy_001_retired_widowed_client',
  'difficulty': 'easy',
  'archetype_id': 'retired_widowed_client',
  'conflicting_objectives': ['estate planning: Wants to review will and beneficiary arrangements.']},
 {'example_id': 'easy_005_inheritance_windfall',
  'difficulty': 'easy',
  'archetype_id': 'inheritance_windfall',
  'conflicting_objectives': ['estate planning: Wants to review beneficiary arrangements and will planning.']},
 {'example_id': 'easy_008_inheritance_windfall',
  'difficulty': 'easy',
  'archetype_id': 'inheritance_windfall',
  'conflicting_objectives': ['estate planning: No will mentioned; wants to make sure the inheritance is protected for the family.']},
 {'example_id': 'easy_010_inheritance_windfall',
  'difficulty': 'easy',
  'archetype_id': 'inheritance_windfall',
  'conflicting_objectives': ['estate planning: No will currently mentioned; considering whether to set one up.']},
 {'example_id': 'easy_018_inheritance_windfall',
  'difficulty': 'easy',
 

In [50]:
conflict_ids = {row['example_id'] for row in obj_conflicts}
conflict_specs = [spec for spec in specs if spec.example_id in conflict_ids]
print(f"{len(conflict_specs)} specs to regenerate")
sorted(conflict_ids)

17 specs to regenerate


['easy_001_retired_widowed_client',
 'easy_005_inheritance_windfall',
 'easy_008_inheritance_windfall',
 'easy_010_inheritance_windfall',
 'easy_018_inheritance_windfall',
 'easy_019_retired_widowed_client',
 'hard_005_young_high_earner_family',
 'hard_010_young_high_earner_family',
 'hard_023_messy_corrections_privacy',
 'hard_025_messy_corrections_privacy',
 'medium_001_self_employed_single',
 'medium_004_self_employed_single',
 'medium_006_messy_corrections_privacy',
 'medium_012_young_high_earner_family',
 'medium_023_high_debt_low_savings',
 'medium_027_dual_income_mortgage_household',
 'medium_028_dual_income_mortgage_household']

In [ ]:
for example_id in conflict_ids:
    path = GROUND_TRUTH_DIR / f"{example_id}.json"
    if path.exists():
        path.unlink()

regen_conflict_packages, regen_conflict_failures = await generate_ground_truths_async(
    async_client,
    conflict_specs,
    max_concurrency=4,
    save=True,
    show_progress=True,
)

print(f"Regenerated: {len(regen_conflict_packages)}  Failures: {len(regen_conflict_failures)}")
regen_conflict_failures

ground_truth:   0%|          | 0/17 [00:00<?, ?example/s]

Regenerated: 17  Failures: 0


[]

In [ ]:
remaining_conflicts = find_objective_section_conflicts()
print(f"{len(remaining_conflicts)} objective-section conflicts remaining")
remaining_conflicts

0 objective-section conflicts remaining


[]

### Schema validation

Verify all saved ground-truth files parse correctly against the `CIFExtraction` schema.


In [8]:
gt_paths = sorted(Path("data/synthetic/ground_truth").glob("*.json"))
len(gt_paths)

80

In [9]:
ground_truth_packages = [json.loads(p.read_text()) for p in gt_paths]

from src.eval.models import CIFExtraction

schema_errors = []

for package in ground_truth_packages:
    try:
        CIFExtraction.model_validate(package["expected"])
    except Exception as exc:
        schema_errors.append({
            "example_id": package["example_id"],
            "error": repr(exc),
        })

len(schema_errors), schema_errors[:3]


(0, [])

### Dataset distribution


In [55]:
from collections import Counter

difficulty_counts = Counter(p["difficulty"] for p in ground_truth_packages)
archetype_counts = Counter(p["archetype_id"] for p in ground_truth_packages)
tag_counts = Counter(tag for p in ground_truth_packages for tag in p["challenge_tags"])
section_counts = Counter(section for p in ground_truth_packages for section in p["section_targets"])

difficulty_counts, archetype_counts, tag_counts, section_counts


(Counter({'hard': 30, 'medium': 30, 'easy': 20}),
 Counter({'inheritance_windfall': 14,
          'pre_retirement_couple': 13,
          'retired_widowed_client': 10,
          'dual_income_mortgage_household': 10,
          'high_debt_low_savings': 10,
          'messy_corrections_privacy': 10,
          'young_high_earner_family': 8,
          'self_employed_single': 5}),
 Counter({'client2_present': 46,
          'objectives_free_form': 44,
          'owner_attribution': 44,
          'numeric_approximate': 43,
          'numeric_exact': 41,
          'correction': 40,
          'negation': 39,
          'joint_assets': 38,
          'estate_planning': 37,
          'advisor_noise': 36,
          'risk_preferences': 36,
          'missing_fields': 34,
          'numeric_shorthand': 32,
          'numeric_range': 32,
          'multiple_products': 25,
          'privacy_reference': 20,
          'date_normalization': 12}),
 Counter({'personal_details': 80,
          'income': 80,
   

In [56]:
def non_empty(value):
    if value is None:
        return False
    if value == []:
        return False
    if isinstance(value, dict):
        return any(non_empty(v) for v in value.values())
    return True

actual_section_counts = Counter()

for package in ground_truth_packages:
    expected = package["expected"]

    if non_empty(expected.get("client1")):
        actual_section_counts["client1"] += 1
    if expected.get("has_client2") or non_empty(expected.get("client2")):
        actual_section_counts["client2"] += 1
    if non_empty(expected.get("household")):
        actual_section_counts["household"] += 1
    if non_empty(expected.get("incomes")):
        actual_section_counts["income"] += 1
    if non_empty(expected.get("expenses")):
        actual_section_counts["expenses"] += 1
    if non_empty(expected.get("pensions_and_retirement_accounts")):
        actual_section_counts["pensions_retirement"] += 1
    if non_empty(expected.get("savings_and_investments")):
        actual_section_counts["savings_investments"] += 1
    if non_empty(expected.get("loans_and_mortgages")):
        actual_section_counts["loans_mortgages"] += 1
    if non_empty(expected.get("other_assets")):
        actual_section_counts["other_assets"] += 1
    if non_empty(expected.get("objectives")):
        actual_section_counts["objectives"] += 1
    if non_empty(expected.get("risk_profile_and_preferences")):
        actual_section_counts["risk_profile_preferences"] += 1
    if non_empty(expected.get("estate_planning")):
        actual_section_counts["estate_planning"] += 1

actual_section_counts


Counter({'client1': 80,
         'income': 80,
         'savings_investments': 80,
         'objectives': 80,
         'household': 77,
         'expenses': 74,
         'client2': 66,
         'risk_profile_preferences': 65,
         'pensions_retirement': 60,
         'loans_mortgages': 58,
         'estate_planning': 47,
         'other_assets': 21})

In [59]:
mismatches = find_inverse_section_mismatches()
len(mismatches), mismatches[:10]

(8,
 [{'example_id': 'easy_001_retired_widowed_client',
   'difficulty': 'easy',
   'archetype_id': 'retired_widowed_client',
   'extra_populated_sections': ['employment']},
  {'example_id': 'easy_008_inheritance_windfall',
   'difficulty': 'easy',
   'archetype_id': 'inheritance_windfall',
   'extra_populated_sections': ['other_assets']},
  {'example_id': 'easy_019_retired_widowed_client',
   'difficulty': 'easy',
   'archetype_id': 'retired_widowed_client',
   'extra_populated_sections': ['employment']},
  {'example_id': 'medium_001_self_employed_single',
   'difficulty': 'medium',
   'archetype_id': 'self_employed_single',
   'extra_populated_sections': ['pensions_retirement']},
  {'example_id': 'medium_004_self_employed_single',
   'difficulty': 'medium',
   'archetype_id': 'self_employed_single',
   'extra_populated_sections': ['pensions_retirement',
    'risk_profile_preferences']},
  {'example_id': 'medium_006_messy_corrections_privacy',
   'difficulty': 'medium',
   'archetype_

In [ ]:
preview_ground_truth_paths = [
    Path("data/synthetic/ground_truth/easy_001_retired_widowed_client.json"),
    Path("data/synthetic/ground_truth/medium_001_self_employed_single.json"),
    Path("data/synthetic/ground_truth/hard_001_retired_widowed_client.json"),
]

In [ ]:
preview_transcript_packages = []
for path in preview_ground_truth_paths:
    ground_truth_package = load_ground_truth_package(path)
    transcript_package = await generate_transcript_for_package(
        async_client,
        ground_truth_package,
    )
    preview_transcript_packages.append(transcript_package)

len(preview_transcript_packages)


3

In [69]:
print(preview_transcript_packages[2]["transcript"])

---
title: N/A
date: 2025-05-12
participants:
  - id: advisor
    name: Megan Carter
  - id: client
    name: Eleanor Fisher
---
ADVISOR: [00:00:03] Hi Eleanor, it’s Megan Carter. Can you hear me okay?

CLIENT1: [00:00:08] Yes, I can. Sorry, I was just... I was looking for my glasses.

ADVISOR: [00:00:12] No problem at all. Thanks for making the time today. I’m just going to ask a bunch of questions so I can get a feel for where things are at and what would actually help. Sound okay?

CLIENT1: [00:00:20] Yes, that’s fine.

ADVISOR: [00:00:23] Great. And just to start with basics, your full name is Eleanor Fisher, right?

CLIENT1: [00:00:28] That’s right.

ADVISOR: [00:00:30] And your date of birth—if you don’t mind me asking?

CLIENT1: [00:00:34] April eighteenth, nineteen fifty-five.

ADVISOR: [00:00:38] Okay, thank you. And are you married, single, widowed...?

CLIENT1: [00:00:44] Widowed. My husband passed a few years ago.

ADVISOR: [00:00:48] I’m sorry. And at home it’s just you no

In [15]:
transcript_packages, transcript_failures = await generate_transcripts_async(
    async_client,
    ground_truth_packages,
    max_concurrency=4,
    save=True,
    show_progress=True,
)

print(f"Generated: {len(transcript_packages)}  Failures: {len(transcript_failures)}")
transcript_failures


transcript:   0%|          | 0/80 [00:00<?, ?example/s]

Generated: 80  Failures: 0


[]

In [8]:
try:
    _packages = transcript_packages
except NameError:
    _packages = [
        load_transcript_package(p)
        for p in sorted(Path("data/synthetic/labels").glob("*.json"))
    ]

In [12]:
transcript_packages = _packages.copy()

In [ ]:
sample_ids = [
    "easy_001_retired_widowed_client",
    "hard_001_retired_widowed_client",
]
sample_packages = [p for p in _packages if p["example_id"] in sample_ids]

sample_validated, sample_val_failures = await validate_examples_async(
    async_client,
    sample_packages,
    max_concurrency=2,
    save=False,
    show_progress=True,
)

print(f"Validated: {len(sample_validated)}  Parse failures: {len(sample_val_failures)}")
for pkg in sample_validated:
    v = pkg["validation"]
    status = "PASS" if v.get("is_valid") else "FAIL"
    reasons = ", ".join(v.get("failure_reasons", [])) or "-"
    print(f"\n  [{status}] {pkg['example_id']}  reasons: {reasons}")
    for leak in v.get("suppression_leaks", []):
        print(f"    suppression_leak — {leak['section']} @ {leak.get('timestamp','?')}: {leak.get('offending_text','')[:100]}")
    for issue in v.get("coverage_issues", []):
        print(f"    coverage_miss    — {issue['field']}: {issue.get('detail','')[:100]}")
    for h in v.get("hallucinations", []):
        print(f"    hallucination    — {h['field']}: gt={h.get('gt_value')} vs transcript={h.get('transcript_value')}")
    for loc in v.get("localization_issues", []):
        print(f"    localization     — '{loc['term']}' @ {loc.get('timestamp','?')}")


validation:   0%|          | 0/2 [00:00<?, ?example/s]

Validated: 2  Parse failures: 0

  [PASS] easy_001_retired_widowed_client  reasons: -

  [PASS] hard_001_retired_widowed_client  reasons: -


In [ ]:
try:
    _all_packages = transcript_packages
except NameError:
    _all_packages = [
        load_transcript_package(p)
        for p in sorted(Path("data/synthetic/labels").glob("*.json"))
    ]

_by_diff = {"easy": [], "medium": [], "hard": []}
for p in _all_packages:
    _by_diff[p["difficulty"]].append(p)

# Pick cases that were NOT in the 2-case smoke test, spread across archetypes
_smoke_ids = {"easy_001_retired_widowed_client", "hard_001_retired_widowed_client"}

def _pick(packages, n, exclude):
    seen_archetypes = set()
    chosen = []
    for p in packages:
        if p["example_id"] in exclude:
            continue
        arch = p.get("archetype_id", "")
        if arch not in seen_archetypes:
            seen_archetypes.add(arch)
            chosen.append(p)
        if len(chosen) == n:
            break
    return chosen

sample5 = (
    _pick(_by_diff["easy"],   1, _smoke_ids)
    + _pick(_by_diff["medium"], 2, _smoke_ids)
    + _pick(_by_diff["hard"],   2, _smoke_ids)
)

print("Running validation on:")
for p in sample5:
    print(f"  {p['example_id']}  [{p['difficulty']}]  archetype={p.get('archetype_id','?')}")

sample5_validated, sample5_failures = await validate_examples_async(
    async_client,
    sample5,
    max_concurrency=2,
    save=False,
    show_progress=True,
)

print(f"\nValidated: {len(sample5_validated)}  Parse failures: {len(sample5_failures)}")
for pkg in sample5_validated:
    v = pkg["validation"]
    status = "PASS" if v.get("is_valid") else "FAIL"
    reasons = ", ".join(v.get("failure_reasons", [])) or "-"
    print(f"\n  [{status}] {pkg['example_id']}  reasons: {reasons}")
    for leak in v.get("suppression_leaks", []):
        print(f"    suppression_leak — {leak['section']} @ {leak.get('timestamp','?')}: {leak.get('offending_text','')[:100]}")
    for issue in v.get("coverage_issues", []):
        print(f"    coverage_miss    — {issue['field']}: {issue.get('detail','')[:100]}")
    for h in v.get("hallucinations", []):
        print(f"    hallucination    — {h['field']}: gt={h.get('gt_value')} vs transcript={h.get('transcript_value')}")
    for loc in v.get("localization_issues", []):
        print(f"    localization     — '{loc['term']}' @ {loc.get('timestamp','?')}")


Running validation on:
  easy_002_dual_income_mortgage_household  [easy]  archetype=dual_income_mortgage_household
  medium_001_self_employed_single  [medium]  archetype=self_employed_single
  medium_002_young_high_earner_family  [medium]  archetype=young_high_earner_family
  hard_002_retired_widowed_client  [hard]  archetype=retired_widowed_client
  hard_003_high_debt_low_savings  [hard]  archetype=high_debt_low_savings


validation:   0%|          | 0/5 [00:00<?, ?example/s]


Validated: 5  Parse failures: 0

  [FAIL] easy_002_dual_income_mortgage_household  reasons: coverage_miss
    coverage_miss    — client1.employment.tax_residency: Tax residency is not mentioned anywhere in the transcript for Lucas.
    coverage_miss    — client2.employment.tax_residency: Tax residency is not mentioned anywhere in the transcript for Patrick.

  [PASS] hard_002_retired_widowed_client  reasons: -

  [PASS] hard_003_high_debt_low_savings  reasons: -

  [PASS] medium_001_self_employed_single  reasons: -

  [PASS] medium_002_young_high_earner_family  reasons: -


In [13]:
validated_packages, validation_failures = await validate_examples_async(
    async_client,
    transcript_packages,
    max_concurrency=2,
    save=True,
    show_progress=True,
)

print(f"Validated: {len(validated_packages)}  Failures: {len(validation_failures)}")
validation_failures


validation:   0%|          | 0/80 [00:00<?, ?example/s]

Validated: 80  Failures: 0


[]

In [ ]:
summary = summarise_validations()

print(f"Total:  {summary['total']}")
print(f"Passed: {summary['passed']}  ({summary['pass_rate']:.0%})")
print(f"Failed: {summary['failed']}")

if summary["failure_reason_counts"]:
    print("\nFailures by reason:")
    for reason, count in sorted(summary["failure_reason_counts"].items(), key=lambda x: -x[1]):
        print(f"  {reason}: {count}")

if summary["failures"]:
    print("\nFailed cases:")
    for f in summary["failures"]:
        reasons = ", ".join(f["failure_reasons"])
        print(f"  {f['example_id']}  [{reasons}]")
        for leak in f["suppression_leaks"]:
            print(f"    suppression_leak — {leak['section']} @ {leak['timestamp']}: {leak['offending_text'][:80]}")
        for issue in f["coverage_issues"]:
            print(f"    coverage_miss    — {issue['field']}: {issue['detail'][:80]}")
        for h in f["hallucinations"]:
            print(f"    hallucination    — {h['field']}: gt={h['gt_value']} transcript={h['transcript_value']}")
        for loc in f["localization_issues"]:
            print(f"    localization     — '{loc['term']}' @ {loc['timestamp']}")

Total:  80
Passed: 28  (35%)
Failed: 52

Failures by reason:
  coverage_miss: 50
  suppression_leak: 14
  hallucination: 2

Failed cases:
  easy_003_high_debt_low_savings  [coverage_miss]
    coverage_miss    — client1.employment.tax_residency: The transcript never explicitly states Connor's tax residency or that he is a US
  easy_004_dual_income_mortgage_household  [coverage_miss, suppression_leak]
    suppression_leak — risk_profile_preferences @ 00:04:04: ADVISOR: [00:04:04] Got it. Do you tend to think of that as longer-term money, o
    coverage_miss    — client1.employment.tax_residency: The transcript never explicitly states Ellie’s tax residency or country; US is o
    coverage_miss    — client2.employment.tax_residency: The transcript never explicitly states Isaac’s tax residency or country; US is o
  easy_005_inheritance_windfall  [coverage_miss]
    coverage_miss    — household.children_or_dependants[0].date_of_birth: The transcript gives Sofia's age (22) and that she is ind

In [ ]:
PROMPT_FIX_IDS = [
    "easy_005_inheritance_windfall",
    "easy_009_dual_income_mortgage_household",
    "easy_018_inheritance_windfall",
    "medium_007_dual_income_mortgage_household",
    "medium_017_dual_income_mortgage_household",
    "medium_028_dual_income_mortgage_household",
]

prompt_fix_packages = [
    load_transcript_package(Path(f"data/synthetic/labels/{eid}.json"))
    for eid in PROMPT_FIX_IDS
]

print(f"Re-validating {len(prompt_fix_packages)} prompt-fixable cases...")

prompt_fix_validated, prompt_fix_failures = await validate_examples_async(
    async_client,
    prompt_fix_packages,
    max_concurrency=2,
    save=True,
    show_progress=True,
)

print(f"\nValidated: {len(prompt_fix_validated)}  Parse failures: {len(prompt_fix_failures)}")
for pkg in prompt_fix_validated:
    v = pkg["validation"]
    status = "PASS" if v.get("is_valid") else "FAIL"
    reasons = ", ".join(v.get("failure_reasons", [])) or "-"
    print(f"\n  [{status}] {pkg['example_id']}  reasons: {reasons}")
    for leak in v.get("suppression_leaks", []):
        print(f"    suppression_leak — {leak['section']} @ {leak.get('timestamp','?')}: {leak.get('offending_text','')[:100]}")
    for issue in v.get("coverage_issues", []):
        print(f"    coverage_miss    — {issue['field']}: {issue.get('detail','')[:100]}")
    for h in v.get("hallucinations", []):
        print(f"    hallucination    — {h['field']}: gt={h.get('gt_value')} vs transcript={h.get('transcript_value')}")
    for loc in v.get("localization_issues", []):
        print(f"    localization     — '{loc['term']}' @ {loc.get('timestamp','?')}")


Re-validating 6 prompt-fixable cases...


validation:   0%|          | 0/6 [00:00<?, ?example/s]


Validated: 6  Parse failures: 0

  [PASS] easy_005_inheritance_windfall  reasons: -

  [PASS] easy_009_dual_income_mortgage_household  reasons: -

  [PASS] easy_018_inheritance_windfall  reasons: -

  [PASS] medium_007_dual_income_mortgage_household  reasons: -

  [PASS] medium_017_dual_income_mortgage_household  reasons: -

  [PASS] medium_028_dual_income_mortgage_household  reasons: -


In [ ]:
_all_summary = summarise_validations()
_failed_ids = {f["example_id"] for f in _all_summary["failures"]}
_regen_ids = sorted(_failed_ids - set(PROMPT_FIX_IDS))

print(f"Cases to regenerate: {len(_regen_ids)}")
for eid in _regen_ids:
    print(f"  {eid}")

# Load ground truth packages for regen cases.
regen_gt_packages = [
    load_ground_truth_package(Path(f"data/synthetic/ground_truth/{eid}.json"))
    for eid in _regen_ids
]


Cases to regenerate: 46
  easy_003_high_debt_low_savings
  easy_004_dual_income_mortgage_household
  easy_006_pre_retirement_couple
  easy_007_pre_retirement_couple
  easy_008_inheritance_windfall
  easy_010_inheritance_windfall
  easy_011_pre_retirement_couple
  easy_014_high_debt_low_savings
  easy_016_pre_retirement_couple
  easy_017_young_high_earner_family
  easy_020_self_employed_single
  hard_004_inheritance_windfall
  hard_005_young_high_earner_family
  hard_008_retired_widowed_client
  hard_009_inheritance_windfall
  hard_010_young_high_earner_family
  hard_011_inheritance_windfall
  hard_013_young_high_earner_family
  hard_014_inheritance_windfall
  hard_015_messy_corrections_privacy
  hard_016_high_debt_low_savings
  hard_017_inheritance_windfall
  hard_019_self_employed_single
  hard_021_messy_corrections_privacy
  hard_022_inheritance_windfall
  hard_024_dual_income_mortgage_household
  hard_025_messy_corrections_privacy
  hard_026_inheritance_windfall
  hard_027_pre_retir

In [ ]:
print(f"\nRegenerating {len(regen_gt_packages)} transcripts...")
regen_transcript_packages, regen_transcript_failures = await generate_transcripts_async(
    async_client,
    regen_gt_packages,
    max_concurrency=4,
    save=True,
    show_progress=True,
)
print(f"Generated: {len(regen_transcript_packages)}  Failures: {len(regen_transcript_failures)}")
if regen_transcript_failures:
    print("Transcript generation failures:", regen_transcript_failures)


print(f"\nValidating {len(regen_transcript_packages)} regenerated transcripts...")
regen_validated, regen_val_failures = await validate_examples_async(
    async_client,
    regen_transcript_packages,
    max_concurrency=2,
    save=True,
    show_progress=True,
)
print(f"Validated: {len(regen_validated)}  Parse failures: {len(regen_val_failures)}")
for pkg in regen_validated:
    v = pkg["validation"]
    status = "PASS" if v.get("is_valid") else "FAIL"
    reasons = ", ".join(v.get("failure_reasons", [])) or "-"
    print(f"\n  [{status}] {pkg['example_id']}  reasons: {reasons}")
    for leak in v.get("suppression_leaks", []):
        print(f"    suppression_leak — {leak['section']} @ {leak.get('timestamp','?')}: {leak.get('offending_text','')[:100]}")
    for issue in v.get("coverage_issues", []):
        print(f"    coverage_miss    — {issue['field']}: {issue.get('detail','')[:100]}")
    for h in v.get("hallucinations", []):
        print(f"    hallucination    — {h['field']}: gt={h.get('gt_value')} vs transcript={h.get('transcript_value')}")
    for loc in v.get("localization_issues", []):
        print(f"    localization     — '{loc['term']}' @ {loc.get('timestamp','?')}")

print("\n--- Overall after regen ---")
final_summary = summarise_validations()
print(f"Total: {final_summary['total']}  Passed: {final_summary['passed']}  Failed: {final_summary['failed']}  ({final_summary['pass_rate']:.0%})")



Regenerating 46 transcripts...


transcript:   0%|          | 0/46 [00:00<?, ?example/s]

Generated: 46  Failures: 0

Validating 46 regenerated transcripts...


validation:   0%|          | 0/46 [00:00<?, ?example/s]

Validated: 46  Parse failures: 0

  [PASS] easy_003_high_debt_low_savings  reasons: -

  [FAIL] easy_004_dual_income_mortgage_household  reasons: coverage_miss
    coverage_miss    — savings_and_investments[0].provider: The transcript mentions a joint high-yield savings account with about $38,000 but never names the ba
    coverage_miss    — savings_and_investments[1].provider: The transcript mentions a joint brokerage account with about $27,000 but does not state the provider
    coverage_miss    — loans_and_mortgages[0].provider: The transcript discusses the mortgage payment, balance, and interest rate but never names the mortga

  [FAIL] easy_006_pre_retirement_couple  reasons: coverage_miss
    coverage_miss    — risk_profile_and_preferences.key_concerns[1]: The transcript mentions an estimated combined Social Security amount (~$52,000 a year) but never dis

  [PASS] easy_007_pre_retirement_couple  reasons: -

  [PASS] easy_008_inheritance_windfall  reasons: -

  [FAIL] easy_010_in

In [ ]:
RISK_CONCERN_IDS = [
    "easy_006_pre_retirement_couple",
    "hard_005_young_high_earner_family",
    "hard_029_pre_retirement_couple",
]

risk_concern_packages = [
    load_transcript_package(Path(f"data/synthetic/labels/{eid}.json"))
    for eid in RISK_CONCERN_IDS
]

print(f"Re-validating {len(risk_concern_packages)} risk-concerns cases...")

rc_validated, rc_parse_failures = await validate_examples_async(
    async_client,
    risk_concern_packages,
    max_concurrency=2,
    save=True,
    show_progress=True,
)

print(f"\nValidated: {len(rc_validated)}  Parse failures: {len(rc_parse_failures)}")
for pkg in rc_validated:
    v = pkg["validation"]
    status = "PASS" if v.get("is_valid") else "FAIL"
    reasons = ", ".join(v.get("failure_reasons", [])) or "-"
    print(f"\n  [{status}] {pkg['example_id']}  reasons: {reasons}")
    for issue in v.get("coverage_issues", []):
        print(f"    coverage_miss — {issue['field']}: {issue.get('detail','')[:100]}")
    for leak in v.get("suppression_leaks", []):
        print(f"    suppression_leak — {leak['section']} @ {leak.get('timestamp','?')}: {leak.get('offending_text','')[:100]}")


Re-validating 3 risk-concerns cases...


validation:   0%|          | 0/3 [00:00<?, ?example/s]


Validated: 3  Parse failures: 0

  [FAIL] easy_006_pre_retirement_couple  reasons: coverage_miss
    coverage_miss — risk_profile_and_preferences.key_concerns[1]: The transcript mentions an estimated combined Social Security benefit ($52,000 a year) but contains 

  [FAIL] hard_005_young_high_earner_family  reasons: coverage_miss
    coverage_miss — risk_profile_and_preferences.key_concerns[4]: The transcript lists joint and individual accounts (401(k)s and a joint Vanguard brokerage) but neve

  [PASS] hard_029_pre_retirement_couple  reasons: -


In [ ]:

TRANSCRIPT_FIX_IDS = [
    # provider name missing
    "easy_004_dual_income_mortgage_household",
    "easy_017_young_high_earner_family",
    "hard_014_inheritance_windfall",
    "hard_021_messy_corrections_privacy",
    "medium_027_dual_income_mortgage_household",
    # DOB not asked
    "easy_014_high_debt_low_savings",
    "hard_013_young_high_earner_family",
    "hard_024_dual_income_mortgage_household",
    "medium_019_young_high_earner_family",
    "medium_020_dual_income_mortgage_household",
    # loan details missing
    "hard_015_messy_corrections_privacy",
    "hard_016_high_debt_low_savings",
    "hard_025_messy_corrections_privacy",
    "hard_028_messy_corrections_privacy",
    "medium_005_dual_income_mortgage_household",
]

tf_gt_packages = [
    load_ground_truth_package(Path(f"data/synthetic/ground_truth/{eid}.json"))
    for eid in TRANSCRIPT_FIX_IDS
]

print(f"Regenerating {len(tf_gt_packages)} transcripts (provider/DOB/loan-details fix)...")
tf_transcript_packages, tf_transcript_failures = await generate_transcripts_async(
    async_client,
    tf_gt_packages,
    max_concurrency=4,
    save=True,
    show_progress=True,
)
print(f"Generated: {len(tf_transcript_packages)}  Failures: {len(tf_transcript_failures)}")

print(f"\nValidating {len(tf_transcript_packages)} regenerated transcripts...")
tf_validated, tf_val_failures = await validate_examples_async(
    async_client,
    tf_transcript_packages,
    max_concurrency=2,
    save=True,
    show_progress=True,
)
print(f"Validated: {len(tf_validated)}  Parse failures: {len(tf_val_failures)}")
for pkg in tf_validated:
    v = pkg["validation"]
    status = "PASS" if v.get("is_valid") else "FAIL"
    reasons = ", ".join(v.get("failure_reasons", [])) or "-"
    print(f"\n  [{status}] {pkg['example_id']}  reasons: {reasons}")
    for leak in v.get("suppression_leaks", []):
        print(f"    suppression_leak — {leak['section']} @ {leak.get('timestamp','?')}: {leak.get('offending_text','')[:100]}")
    for issue in v.get("coverage_issues", []):
        print(f"    coverage_miss — {issue['field']}: {issue.get('detail','')[:100]}")
    for h in v.get("hallucinations", []):
        print(f"    hallucination — {h['field']}: gt={h.get('gt_value')} vs transcript={h.get('transcript_value')}")


Regenerating 15 transcripts (provider/DOB/loan-details fix)...


transcript:   0%|          | 0/15 [00:00<?, ?example/s]

Generated: 15  Failures: 0

Validating 15 regenerated transcripts...


validation:   0%|          | 0/15 [00:00<?, ?example/s]

Validated: 15  Parse failures: 0

  [PASS] easy_004_dual_income_mortgage_household  reasons: -

  [PASS] easy_014_high_debt_low_savings  reasons: -

  [FAIL] easy_017_young_high_earner_family  reasons: coverage_miss
    coverage_miss — expenses[1].amount.normalized_amount: Label includes a household mortgage payment expense of $3,800 per month, but the transcript never me

  [PASS] hard_013_young_high_earner_family  reasons: -

  [FAIL] hard_014_inheritance_windfall  reasons: coverage_miss
    coverage_miss — client1.personal.email: Client 1's email address is not mentioned anywhere in the transcript.

  [FAIL] hard_015_messy_corrections_privacy  reasons: coverage_miss
    coverage_miss — loans_and_mortgages[1].provider: The provider of Brian's car loan is never mentioned in the transcript; only the payment amount and a

  [PASS] hard_016_high_debt_low_savings  reasons: -

  [FAIL] hard_021_messy_corrections_privacy  reasons: coverage_miss
    coverage_miss — loans_and_mortgages[1].pro

In [ ]:
PURE_REGEN_IDS = [
    # suppression leaks
    "hard_004_inheritance_windfall",
    "hard_009_inheritance_windfall",
    "hard_022_inheritance_windfall",
    "medium_002_young_high_earner_family",
    "medium_014_high_debt_low_savings",
    "medium_018_high_debt_low_savings",
    # seriously incomplete transcripts
    "easy_010_inheritance_windfall",
    "hard_026_inheritance_windfall",
    "medium_021_pre_retirement_couple",
    # other coverage gaps (expense sections, asset sections, single missing items)
    "hard_010_young_high_earner_family",
    "hard_011_inheritance_windfall",
    "medium_003_retired_widowed_client",
    "medium_006_messy_corrections_privacy",
    "medium_008_pre_retirement_couple",
]

pr_gt_packages = [
    load_ground_truth_package(Path(f"data/synthetic/ground_truth/{eid}.json"))
    for eid in PURE_REGEN_IDS
]

print(f"Regenerating {len(pr_gt_packages)} transcripts (suppression leaks / incomplete / other)...")
pr_transcript_packages, pr_transcript_failures = await generate_transcripts_async(
    async_client,
    pr_gt_packages,
    max_concurrency=4,
    save=True,
    show_progress=True,
)
print(f"Generated: {len(pr_transcript_packages)}  Failures: {len(pr_transcript_failures)}")

print(f"\nValidating {len(pr_transcript_packages)} regenerated transcripts...")
pr_validated, pr_val_failures = await validate_examples_async(
    async_client,
    pr_transcript_packages,
    max_concurrency=2,
    save=True,
    show_progress=True,
)
print(f"Validated: {len(pr_validated)}  Parse failures: {len(pr_val_failures)}")
for pkg in pr_validated:
    v = pkg["validation"]
    status = "PASS" if v.get("is_valid") else "FAIL"
    reasons = ", ".join(v.get("failure_reasons", [])) or "-"
    print(f"\n  [{status}] {pkg['example_id']}  reasons: {reasons}")
    for leak in v.get("suppression_leaks", []):
        print(f"    suppression_leak — {leak['section']} @ {leak.get('timestamp','?')}: {leak.get('offending_text','')[:100]}")
    for issue in v.get("coverage_issues", []):
        print(f"    coverage_miss — {issue['field']}: {issue.get('detail','')[:100]}")
    for h in v.get("hallucinations", []):
        print(f"    hallucination — {h['field']}: gt={h.get('gt_value')} vs transcript={h.get('transcript_value')}")

print("\n--- Final overall summary ---")
final_summary = summarise_validations()
print(f"Total: {final_summary['total']}  Passed: {final_summary['passed']}  Failed: {final_summary['failed']}  ({final_summary['pass_rate']:.0%})")
if final_summary["failure_reason_counts"]:
    print("Remaining failures by reason:")
    for reason, count in sorted(final_summary["failure_reason_counts"].items(), key=lambda x: -x[1]):
        print(f"  {reason}: {count}")


Regenerating 14 transcripts (suppression leaks / incomplete / other)...


transcript:   0%|          | 0/14 [00:00<?, ?example/s]

Generated: 14  Failures: 0

Validating 14 regenerated transcripts...


validation:   0%|          | 0/14 [00:00<?, ?example/s]

Validated: 14  Parse failures: 0

  [PASS] easy_010_inheritance_windfall  reasons: -

  [FAIL] hard_004_inheritance_windfall  reasons: coverage_miss
    coverage_miss — client1.employment.desired_retirement_age: Desired retirement age is never discussed in the transcript.
    coverage_miss — other_assets[0].description: The client mentions a monthly home payment but never states that they own a primary residence as an 
    coverage_miss — other_assets[0].current_value.normalized_amount: No value or price range is given for the primary residence in the transcript.
    coverage_miss — other_assets[0].current_value.lower_bound: No value or price range is given for the primary residence in the transcript.
    coverage_miss — other_assets[0].current_value.upper_bound: No value or price range is given for the primary residence in the transcript.
    coverage_miss — other_assets[0].current_value.is_approximate: The transcript does not provide any approximate valuation wording (e.g. 'about' or

In [10]:
final_summary["failures"]

[{'example_id': 'easy_006_pre_retirement_couple',
  'failure_reasons': ['coverage_miss'],
  'coverage_issues': [{'field': 'risk_profile_and_preferences.key_concerns[1]',
    'expected_value': 'Making sure Social Security timing fits retirement income needs',
    'detail': 'The transcript mentions an estimated combined Social Security benefit ($52,000 a year) but contains no discussion or expressed concern about when to claim Social Security or how its timing should align with retirement income needs.'}],
  'suppression_leaks': [],
  'hallucinations': [],
  'localization_issues': []},
 {'example_id': 'easy_017_young_high_earner_family',
  'failure_reasons': ['coverage_miss'],
  'coverage_issues': [{'field': 'expenses[1].amount.normalized_amount',
    'expected_value': '3800.0',
    'detail': 'Label includes a household mortgage payment expense of $3,800 per month, but the transcript never mentions any mortgage or housing payment amount.'}],
  'suppression_leaks': [],
  'hallucinations':

In [ ]:
ROUND3_IDS = [
    "easy_006_pre_retirement_couple",
    "easy_017_young_high_earner_family",
    "hard_004_inheritance_windfall",
    "hard_005_young_high_earner_family",
    "hard_009_inheritance_windfall",
    "hard_011_inheritance_windfall",
    "hard_014_inheritance_windfall",
    "hard_015_messy_corrections_privacy",
    "hard_021_messy_corrections_privacy",
    "hard_022_inheritance_windfall",
    "hard_024_dual_income_mortgage_household",
    "hard_025_messy_corrections_privacy",
    "hard_026_inheritance_windfall",
    "hard_028_messy_corrections_privacy",
    "medium_006_messy_corrections_privacy",
    "medium_008_pre_retirement_couple",
    "medium_014_high_debt_low_savings",
    "medium_018_high_debt_low_savings",
    "medium_021_pre_retirement_couple",
]

r3_gt_packages = [
    load_ground_truth_package(Path(f"data/synthetic/ground_truth/{eid}.json"))
    for eid in ROUND3_IDS
]

print(f"Regenerating {len(r3_gt_packages)} transcripts...")
r3_transcripts, r3_transcript_failures = await generate_transcripts_async(
    async_client, r3_gt_packages, max_concurrency=4, save=True, show_progress=True,
)
print(f"Generated: {len(r3_transcripts)}  Failures: {len(r3_transcript_failures)}")

print(f"\nValidating {len(r3_transcripts)} transcripts...")
r3_validated, r3_val_failures = await validate_examples_async(
    async_client, r3_transcripts, max_concurrency=2, save=True, show_progress=True,
)
print(f"Validated: {len(r3_validated)}  Parse failures: {len(r3_val_failures)}")
for pkg in r3_validated:
    v = pkg["validation"]
    status = "PASS" if v.get("is_valid") else "FAIL"
    reasons = ", ".join(v.get("failure_reasons", [])) or "-"
    print(f"\n  [{status}] {pkg['example_id']}  reasons: {reasons}")
    for leak in v.get("suppression_leaks", []):
        print(f"    suppression_leak  {leak['section']} @ {leak.get('timestamp','?')}: {leak.get('offending_text','')[:100]}")
    for issue in v.get("coverage_issues", []):
        print(f"    coverage_miss     {issue['field']}: {issue.get('detail','')[:100]}")
    for h in v.get("hallucinations", []):
        print(f"    hallucination     {h['field']}: gt={h.get('gt_value')} vs {str(h.get('transcript_value',''))[:80]}")

print("\n--- Overall after round 3 ---")
r3_summary = summarise_validations()
print(f"Total: {r3_summary['total']}  Passed: {r3_summary['passed']}  Failed: {r3_summary['failed']}  ({r3_summary['pass_rate']:.0%})")
if r3_summary["failure_reason_counts"]:
    print("Remaining failures by reason:")
    for reason, count in sorted(r3_summary["failure_reason_counts"].items(), key=lambda x: -x[1]):
        print(f"  {reason}: {count}")


Regenerating 19 transcripts...


transcript:   0%|          | 0/19 [00:00<?, ?example/s]

Generated: 19  Failures: 0

Validating 19 transcripts...


validation:   0%|          | 0/19 [00:00<?, ?example/s]

Validated: 19  Parse failures: 0

  [PASS] easy_006_pre_retirement_couple  reasons: -

  [FAIL] easy_017_young_high_earner_family  reasons: suppression_leak
    suppression_leak  loans_mortgages @ 00:03:12: CLIENT1: [00:03:12] Sure. Mortgage is $3,800 a month.

  [FAIL] hard_004_inheritance_windfall  reasons: coverage_miss
    coverage_miss     client1.employment.desired_retirement_age: No retirement age or retirement timing is discussed in the transcript, so there is no support for a 

  [FAIL] hard_005_young_high_earner_family  reasons: suppression_leak
    suppression_leak  loans_mortgages @ 00:04:01: CLIENT1: [00:04:01] The mortgage is 4,100 a month.

  [FAIL] hard_009_inheritance_windfall  reasons: suppression_leak
    suppression_leak  pensions_retirement @ 00:06:58: CLIENT2: [00:06:58] I’ve got a Roth IRA — it’s about 39k.

  [PASS] hard_011_inheritance_windfall  reasons: -

  [PASS] hard_014_inheritance_windfall  reasons: -

  [FAIL] hard_015_messy_corrections_privacy  reasons:

In [7]:
summarise_validations()["failures"]

[{'example_id': 'easy_017_young_high_earner_family',
  'failure_reasons': ['suppression_leak'],
  'coverage_issues': [],
  'suppression_leaks': [{'section': 'loans_mortgages',
    'timestamp': '00:03:12',
    'offending_text': 'CLIENT1: [00:03:12] Sure. Mortgage is $3,800 a month.'}],
  'hallucinations': [],
  'localization_issues': []},
 {'example_id': 'hard_004_inheritance_windfall',
  'failure_reasons': ['coverage_miss'],
  'coverage_issues': [{'field': 'client1.employment.desired_retirement_age',
    'expected_value': '65',
    'detail': 'No retirement age or retirement timing is discussed in the transcript, so there is no support for a desired retirement age of 65.'}],
  'suppression_leaks': [],
  'hallucinations': [],
  'localization_issues': []},
 {'example_id': 'hard_005_young_high_earner_family',
  'failure_reasons': ['suppression_leak'],
  'coverage_issues': [],
  'suppression_leaks': [{'section': 'loans_mortgages',
    'timestamp': '00:04:01',
    'offending_text': 'CLIENT1:

In [ ]:
ROUND4_IDS = [
    "easy_017_young_high_earner_family",
    "hard_004_inheritance_windfall",
    "hard_005_young_high_earner_family",
    "hard_009_inheritance_windfall",
    "hard_015_messy_corrections_privacy",
    "hard_022_inheritance_windfall",
    "hard_024_dual_income_mortgage_household",
    "hard_025_messy_corrections_privacy",
    "hard_026_inheritance_windfall",
    "medium_006_messy_corrections_privacy",
    "medium_008_pre_retirement_couple",
    "medium_014_high_debt_low_savings",
    "medium_018_high_debt_low_savings",
]

r4_gt_packages = [
    load_ground_truth_package(Path(f"data/synthetic/ground_truth/{eid}.json"))
    for eid in ROUND4_IDS
]

print(f"Regenerating {len(r4_gt_packages)} transcripts...")
r4_transcripts, r4_transcript_failures = await generate_transcripts_async(
    async_client, r4_gt_packages, max_concurrency=4, save=True, show_progress=True,
)
print(f"Generated: {len(r4_transcripts)}  Failures: {len(r4_transcript_failures)}")

print(f"\nValidating {len(r4_transcripts)} transcripts...")
r4_validated, r4_val_failures = await validate_examples_async(
    async_client, r4_transcripts, max_concurrency=2, save=True, show_progress=True,
)
print(f"Validated: {len(r4_validated)}  Parse failures: {len(r4_val_failures)}")
for pkg in r4_validated:
    v = pkg["validation"]
    status = "PASS" if v.get("is_valid") else "FAIL"
    reasons = ", ".join(v.get("failure_reasons", [])) or "-"
    print(f"\n  [{status}] {pkg['example_id']}  reasons: {reasons}")
    for leak in v.get("suppression_leaks", []):
        print(f"    suppression_leak  {leak['section']} @ {leak.get('timestamp','?')}: {leak.get('offending_text','')[:120]}")
    for issue in v.get("coverage_issues", []):
        print(f"    coverage_miss     {issue['field']}: {issue.get('detail','')[:120]}")
    for h in v.get("hallucinations", []):
        print(f"    hallucination     {h['field']}: gt={h.get('gt_value')} vs {str(h.get('transcript_value',''))[:80]}")

print("\n--- Overall after round 4 ---")
r4_summary = summarise_validations()
print(f"Total: {r4_summary['total']}  Passed: {r4_summary['passed']}  Failed: {r4_summary['failed']}  ({r4_summary['pass_rate']:.0%})")
if r4_summary["failure_reason_counts"]:
    print("Remaining failures by reason:")
    for reason, count in sorted(r4_summary["failure_reason_counts"].items(), key=lambda x: -x[1]):
        print(f"  {reason}: {count}")
if r4_summary["failures"]:
    print("\nFailed cases:")
    for f in r4_summary["failures"]:
        print(f"  {f['example_id']}  [{', '.join(f['failure_reasons'])}]")


Regenerating 13 transcripts...


transcript:   0%|          | 0/13 [00:00<?, ?example/s]

Generated: 13  Failures: 0

Validating 13 transcripts...


validation:   0%|          | 0/13 [00:00<?, ?example/s]

Validated: 13  Parse failures: 0

  [FAIL] easy_017_young_high_earner_family  reasons: coverage_miss
    coverage_miss     pensions_and_retirement_accounts[0].account_type: The transcript refers to Grace's plan only as a 'work retirement plan' / 'Northstar plan' and never explicitly identifie
    coverage_miss     pensions_and_retirement_accounts[1].account_type: The transcript refers to Natalie's plan only as a workplace/ BrightWave Media retirement plan and never explicitly ident

  [FAIL] hard_004_inheritance_windfall  reasons: coverage_miss, suppression_leak
    suppression_leak  loans_mortgages @ 00:02:50: Mortgage payment is $2,450 a month.
    coverage_miss     other_assets[0].current_value.normalized_amount: Value of the primary residence is never mentioned in the transcript.
    coverage_miss     other_assets[0].current_value.lower_bound: No range or lower-bound value for the primary residence is discussed.
    coverage_miss     other_assets[0].current_value.upper_bound: No ra